# Table 1 — Structural Collapse of TPC-DS at Scale

**Problem statement, in one small table.** `dsqgen` can produce as many TPC-DS
queries as you like by re-instantiating its 99 templates with new substitution
parameters, but every instance of a template still asks Trino's optimizer to
do the same shape of work. This notebook checks how much of that
"more queries" is actually "more structure", using three query sets generated
at increasing scale from the *same* 99 templates:

- `Workloads/tpcds` — the official 99-query set (one instance per template)
- `Workloads/tpcds_1000` — 1,000 instances (~10/template)
- `Workloads/TPCDS_10000` — 10,000 instances (~100/template)

This is a focused, publication-ready version of section 7
("Problem statement — structural collapse at scale") of
`compare_workloads.ipynb`: same primitive (`canonical_plan_graph_signature`
from `loader/workload_analysis.py`), reduced to the handful of numbers that
make the point, with columns chosen so a reader gets the headline finding —
**structural collapse is already present at the official 99-query scale, and
gets categorically worse from there** — without needing to parse a ratio.

**Offline by design:** reads each workload's cached `plans.json`
(written by `capture_query_plans(...)` in `analysis.py`) — no live Trino
connection needed. Run that capture first for any workload missing a
`plans.json`.

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd

from analysis import resolve_workload_path, load_plans_file
from loader.workload_analysis import dags_from_plan_bundle, canonical_plan_graph_signature

WORKLOAD_ROOT = Path("/mnt/primary/Main/Workloads")
RESULTS_DIR = Path("results")

# Same three query sets as compare_workloads.ipynb section 7 -- the official
# 99-template TPC-DS set, generated by the same `dsqgen` generator at
# increasing scale. Order matters: rows print smallest -> largest.
SCALE_WORKLOADS = ["tpcds", "tpcds_1000", "TPCDS_10000"]
SCALE_LABEL = {"tpcds": "99 (official)", "tpcds_1000": "1,000", "TPCDS_10000": "10,000"}

## Metric design

The headline metric is **coverage loss**: how many queries in the set fail to
contribute any structure beyond what another query already contributes. Four
columns:

- **Distinct Plan Shapes** — number of structurally-unique Trino EXPLAIN plan
  graphs (`canonical_plan_graph_signature`), i.e. how much real structural
  coverage the query set actually achieves, independent of how large *N* is.
- **Collapsed Queries** — number of queries whose plan shape is *shared with
  at least one other query* in the set, i.e. queries that add zero coverage
  beyond a query already present. (A query is *not* collapsed only if its
  plan shape is unique to it within the set.)
- **Collapsed (%)** — the same count as a share of the query set, so the 99,
  1,000 and 10,000-query rows are comparable at a glance.
- **Largest Duplicate Group** — how many queries collapse onto the single
  *most*-repeated plan shape, i.e. the worst-case concrete instance of
  collapse, not just an aggregate rate.

In [2]:
def structural_collapse_row(workload_name):
    path = resolve_workload_path(workload_name, WORKLOAD_ROOT)
    bundle = load_plans_file(path)
    if bundle is None:
        raise FileNotFoundError(
            f"No plans.json for '{workload_name}' at {path}. Run capture_query_plans(...) "
            "first for this workload."
        )

    dags = dags_from_plan_bundle(bundle)
    n = len(dags)

    shape_counts = Counter(canonical_plan_graph_signature(dags[q]) for q in dags)
    n_shapes = len(shape_counts)
    # A query "collapses" if its plan shape is shared with >=1 other query --
    # i.e. it adds no structural coverage the set doesn't already have.
    collapsed = sum(c for c in shape_counts.values() if c > 1)

    return {
        "TPC-DS Query Set": SCALE_LABEL[workload_name],
        "N": n,
        "Distinct Plan Shapes": n_shapes,
        "Collapsed Queries": collapsed,
        "Collapsed (%)": round(100 * collapsed / n, 1),
        "Largest Duplicate Group": max(shape_counts.values()),
    }

table1_df = pd.DataFrame(
    [structural_collapse_row(w) for w in SCALE_WORKLOADS]
).set_index("TPC-DS Query Set")

table1_df

,N,Distinct Plan Shapes,Collapsed Queries,Collapsed (%),Largest Duplicate Group
TPC-DS Query Set,,,,,
99 (official),99,84,27,27.3,4
"1,000",1000,87,1000,100.0,39
"10,000",10000,88,10000,100.0,388


Sanity check: recompute `Collapsed Queries` a second, independent way -- as
`N` minus the number of plan shapes that appear *exactly once* -- and confirm
it agrees with the direct sum used above. The two formulas only agree if
every collapsed/non-collapsed query is classified consistently.

In [3]:
for wname in SCALE_WORKLOADS:
    lbl = SCALE_LABEL[wname]
    bundle = load_plans_file(resolve_workload_path(wname, WORKLOAD_ROOT))
    dags = dags_from_plan_bundle(bundle)
    shape_counts = Counter(canonical_plan_graph_signature(dags[q]) for q in dags)
    singleton_shapes = sum(1 for c in shape_counts.values() if c == 1)
    expected_collapsed = len(dags) - singleton_shapes
    assert table1_df.at[lbl, "Collapsed Queries"] == expected_collapsed, lbl

row99 = SCALE_LABEL["tpcds"]
row1k = SCALE_LABEL["tpcds_1000"]
row10k = SCALE_LABEL["TPCDS_10000"]
print(
    f"Even the official {table1_df.at[row99, 'N']}-query TPC-DS set already has "
    f"{table1_df.at[row99, 'Collapsed Queries']} queries "
    f"({table1_df.at[row99, 'Collapsed (%)']}%) that share their Trino plan with "
    "another query in the set -- more than one in four queries add no new "
    "structural coverage.\n"
    f"At {table1_df.at[row1k, 'N']:,} generated queries, EVERY query collapses this way "
    f"({table1_df.at[row1k, 'Collapsed (%)']}%) -- coverage tops out almost immediately.\n"
    f"Pushing to {table1_df.at[row10k, 'N']:,} queries doesn't move that number at all "
    f"({table1_df.at[row10k, 'Collapsed (%)']}%): all {table1_df.at[row10k, 'N']:,} queries "
    f"still land on just {table1_df.at[row10k, 'Distinct Plan Shapes']} distinct plan shapes, "
    f"with the single most-repeated shape alone accounting for "
    f"{table1_df.at[row10k, 'Largest Duplicate Group']} queries."
)

Even the official 99-query TPC-DS set already has 27 queries (27.3%) that share their Trino plan with another query in the set -- more than one in four queries add no new structural coverage.
At 1,000 generated queries, EVERY query collapses this way (100.0%) -- coverage tops out almost immediately.
Pushing to 10,000 queries doesn't move that number at all (100.0%): all 10,000 queries still land on just 88 distinct plan shapes, with the single most-repeated shape alone accounting for 388 queries.


## LaTeX export

`N` is dropped from the printed table -- it's already carried by the row
label ("99 (official)" / "1,000" / "10,000"), and keeping the LaTeX table to
four columns is the whole point of this notebook.

In [4]:
latex_df = table1_df.drop(columns="N").rename(columns={"Collapsed (%)": "Collapsed (\\%)"})
latex_df.index.name = None  # drop the index name so to_latex doesn't emit a blank header row

latex_str = latex_df.to_latex(
    float_format="%.1f",
    caption=(
        "Structural coverage loss in TPC-DS query plans across generation scale. "
        "Plan shapes are counted by exact Trino EXPLAIN plan-graph match "
        "(\\texttt{canonical\\_plan\\_graph\\_signature}); a query is `collapsed' if its plan "
        "shape is shared with another query in the set, i.e. it adds no structural coverage "
        "beyond what the set already has. Over a quarter of the official 99-query set is "
        "already collapsed, and every query collapses by 1,000 queries -- scaling generation "
        "up a further order of magnitude adds no additional coverage, only more copies of the "
        "same $\\sim$88 plan shapes."
    ),
    label="tab:tpcds-structural-collapse",
)
print(latex_str)

RESULTS_DIR.mkdir(exist_ok=True)
table1_df.to_csv(RESULTS_DIR / "table1_structural_collapse.csv")
(RESULTS_DIR / "table1_structural_collapse.tex").write_text(latex_str)
print(f"wrote {RESULTS_DIR / 'table1_structural_collapse.csv'}")
print(f"wrote {RESULTS_DIR / 'table1_structural_collapse.tex'}")

\begin{table}
\caption{Structural coverage loss in TPC-DS query plans across generation scale. Plan shapes are counted by exact Trino EXPLAIN plan-graph match (\texttt{canonical\_plan\_graph\_signature}); a query is `collapsed' if its plan shape is shared with another query in the set, i.e. it adds no structural coverage beyond what the set already has. Over a quarter of the official 99-query set is already collapsed, and every query collapses by 1,000 queries -- scaling generation up a further order of magnitude adds no additional coverage, only more copies of the same $\sim$88 plan shapes.}
\label{tab:tpcds-structural-collapse}
\begin{tabular}{lrrrr}
\toprule
 & Distinct Plan Shapes & Collapsed Queries & Collapsed (\%) & Largest Duplicate Group \\
\midrule
99 (official) & 84 & 27 & 27.3 & 4 \\
1,000 & 87 & 1000 & 100.0 & 39 \\
10,000 & 88 & 10000 & 100.0 & 388 \\
\bottomrule
\end{tabular}
\end{table}



wrote results/table1_structural_collapse.csv
wrote results/table1_structural_collapse.tex
